In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

np.random.seed(42)
N_LOANS = 10000

In [3]:
loan_id = np.arange(1, N_LOANS + 1)
loan_type = np.random.choice(['term', 'revolving'], size=N_LOANS, p=[0.7, 0.3])
region = np.random.choice(['North', 'South', 'East', 'West'], size=N_LOANS)
loan_purpose = np.random.choice(['auto', 'personal', 'home_improvement', 'debt_consolidation'], size=N_LOANS)

origination_date = pd.to_datetime('2021-01-01') + pd.to_timedelta(
    np.random.randint(0, 1460, N_LOANS), unit='D')

orig_credit_score = np.random.normal(680, 60, N_LOANS).clip(300, 850).astype(int)
original_balance = np.random.gamma(shape=2, scale=8000, size=N_LOANS).clip(1000, 100000).round(2)
interest_rate = np.random.normal(9, 3, N_LOANS).clip(3, 25).round(2)

In [7]:
risk_factor = (750 - orig_credit_score) / 450

score_drift = np.random.normal(-20, 15, N_LOANS) - (risk_factor * 40)
current_credit_score = (orig_credit_score + score_drift).clip(300, 850).astype(int)

dpd_lambda = np.clip(2 + risk_factor * 40, 0.1, None)
days_past_due = np.random.poisson(dpd_lambda, N_LOANS).clip(0, 180)

watchlist_flag = (np.random.rand(N_LOANS) < (0.05 + risk_factor * 0.25)).astype(int)

default_prob = np.clip((days_past_due / 200) + (risk_factor * 0.15), 0, 0.98)
default_flag = (np.random.rand(N_LOANS) < default_prob).astype(int)
default_flag[days_past_due >= 90] = 1

In [12]:
months_since_orig = np.clip((pd.Timestamp('2025-01-01') - origination_date).days / 30, 1, None)
paydown_pct = np.clip(months_since_orig * np.random.uniform(0.01, 0.03, N_LOANS), 0, 0.9)
current_balance = (original_balance * (1 - paydown_pct)).round(2)

credit_limit = np.where(
    loan_type == 'revolving',
    (current_balance * np.random.uniform(1.5, 3, N_LOANS)).round(2),
    np.nan
)

In [13]:
df = pd.DataFrame({
    'loan_id': loan_id,
    'loan_type': loan_type,
    'region': region,
    'loan_purpose': loan_purpose,
    'origination_date': origination_date,
    'orig_credit_score': orig_credit_score,
    'current_credit_score': current_credit_score,
    'original_balance': original_balance,
    'current_balance': current_balance,
    'credit_limit': credit_limit,
    'interest_rate': interest_rate,
    'days_past_due': days_past_due,
    'watchlist_flag': watchlist_flag,
    'default_flag': default_flag,
})

df.to_csv('../data/raw/synthetic_loan_book.csv', index=False)
print(df.shape)
df.head()

(10000, 14)


,loan_id,loan_type,region,loan_purpose,origination_date,orig_credit_score,current_credit_score,original_balance,current_balance,credit_limit,interest_rate,days_past_due,watchlist_flag,default_flag
0,1,term,East,personal,2024-12-25,633,617,7835.88,7697.92,NaN,16.96,9,0,0
1,2,revolving,East,personal,2024-10-05,727,713,18202.97,17050.19,36929.54,9.82,1,0,0
2,3,term,South,auto,2023-08-02,625,600,19197.03,14505.45,NaN,9.65,5,0,0
3,4,term,South,auto,2022-12-19,613,565,8637.39,4821.45,NaN,7.25,14,0,0
4,5,term,West,personal,2022-08-17,752,732,19720.10,3186.66,NaN,7.02,2,1,0


In [14]:
print(df.groupby(pd.cut(df['orig_credit_score'], bins=[300,600,650,700,750,850]))['default_flag'].mean())
print(df.groupby(pd.cut(df['days_past_due'], bins=[-1,0,30,60,90,180]))['default_flag'].mean())
print(df['default_flag'].mean())

orig_credit_score
(300, 600]    0.122056
(600, 650]    0.107542
(650, 700]    0.067242
(700, 750]    0.029304
(750, 850]    0.001641
Name: default_flag, dtype: float64
days_past_due
(-1, 0]     0.000000
(0, 30]     0.069806
(30, 60]    0.166667
Name: default_flag, dtype: float64
0.0637


# Synthetic Loan Book Generation

This notebook generates a synthetic retail loan portfolio of 10,000 loans for the IFRS 9 ECL Engine project.
Fields are deliberately correlated (e.g. lower origination credit score -> higher days-past-due and default
probability) to simulate realistic credit risk relationships, since real bank loan-tape data is not public.

Key fields:
- orig_credit_score vs current_credit_score: used later to detect Significant Increase in Credit Risk (SICR)
- days_past_due: primary staging trigger (30+ -> Stage 2, 90+ -> Stage 3)
- loan_type / credit_limit: distinguishes term vs revolving exposures for EAD calculation